# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya: Demonstrating AI-Ready Data Standards for Data Infrastructure in Africa Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² Mental Health Survey Dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
# Print dataset name and description
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will enumerate record sets defined by the dataset and list their fields and columns by `@id`.

In [ ]:
record_sets = list(dataset.record_sets())
print(f"Found {len(record_sets)} record sets:")
for rs in record_sets:
    print(f"- Record Set Name: {rs.name}, @id: {rs['@id']}")
    print("  Fields:")
    for field in rs.fields:
        print(f"    - Field Name: {field.name}, @id: {field['@id']}, Data Type: {getattr(field, 'data_type', 'N/A')}")
    print("  Columns:")
    for col in rs.columns:
        print(f"    - Column Name: {col.name}, @id: {col['@id']}, Data Type: {getattr(col, 'data_type', 'N/A')}")
    print("  Example Record:")
    records = list(dataset.records(record_set=rs['@id']))
    if records:
        print(records[0])
    else:
        print("    (No records loaded)")
    print()

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Below, we extract each record set using their `@id`:

In [ ]:
# List the record set @ids found above
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

# Load all record sets into DataFrames
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded DataFrame for Record Set @id {record_set_id}: {df.shape}")
    print(f"Columns (@id): {df.columns.tolist()}")
    print(df.head(), "\n")
# For demonstration, we'll select the first record set to focus further EDA
main_record_set_id = record_set_ids[0]

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

Let's:
- Select a numeric field (e.g., GAD-7 score or PHQ-9 score) by its field `@id`
- Apply a filter for high scores
- Normalize the score
- Group by demographic attribute (e.g., gender or education) by their `@id`

In [ ]:
# Review the columns of the main record set
main_df = dataframes[main_record_set_id]
print("Columns in main record set:", main_df.columns.tolist())

# Select field @ids for GAD-7 score, gender, and education; adjust below as required
gad7_field_id = None
gender_field_id = None
education_field_id = None
for col in main_df.columns:
    if 'gad' in col.lower() or 'GAD' in col:
        gad7_field_id = col
    elif 'gender' in col.lower():
        gender_field_id = col
    elif 'education' in col.lower():
        education_field_id = col
print(f"GAD-7 field @id: {gad7_field_id}, Gender field @id: {gender_field_id}, Education field @id: {education_field_id}")

# EDA (example threshold for GAD-7 score)
threshold = 10
if gad7_field_id:
    filtered_df = main_df[main_df[gad7_field_id].astype(float) > threshold]
    print(f"Filtered records with {gad7_field_id} > {threshold}:")
    print(filtered_df.head())

    # Normalize the numeric field
    filtered_df[f"{gad7_field_id}_normalized"] = (
        filtered_df[gad7_field_id].astype(float) - filtered_df[gad7_field_id].astype(float).mean()
    ) / filtered_df[gad7_field_id].astype(float).std()
    print(f"Normalized {gad7_field_id} for filtered records:")
    print(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

    # Group by gender
    group_field_id = gender_field_id if gender_field_id else education_field_id
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[gad7_field_id].mean().reset_index()
        print(f"Grouped avg {gad7_field_id} by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize distribution of GAD-7 scores and their relation to gender or education level.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_field_id:
    plt.figure(figsize=(8, 5))
    sns.histplot(main_df[gad7_field_id].dropna().astype(float), bins=10, kde=True, color='skyblue')
    plt.title(f"Distribution of {gad7_field_id} Scores")
    plt.xlabel("Score")
    plt.ylabel("Count")
    plt.show()

    if gender_field_id:
        plt.figure(figsize=(8, 5))
        sns.boxplot(x=main_df[gender_field_id], y=main_df[gad7_field_id].astype(float))
        plt.title(f"{gad7_field_id} Score by {gender_field_id}")
        plt.xlabel("Gender")
        plt.ylabel("GAD-7 Score")
        plt.show()

## 6. Conclusion
We successfully loaded the Kilifi County FAIR² Mental Health Survey Dataset using `mlcroissant`.
- We explored record sets and fields using their `@id` references.
- Performed EDA and visualized key mental health indicators (GAD-7).
- Grouped results by demographic attributes to inform public health analysis.

This approach supports transparent, reproducible, and AI-ready FAIR data exploration with Croissant schemas.